# SatQuery — backend on Colab**Runtime → Change runtime type → T4 GPU** before running anything.Run the cells in order. Cell 5 prints the public URL; paste it into`frontend/.env` as `VITE_API_URL`, set `VITE_MOCK=0`, restart Vite.Buy Colab Pro. The free tier disconnects at roughly 90 minutes idle and this isnot the place to save the money.

In [ ]:
# 1 · install!pip install -q fastapi uvicorn nest-asyncio pyngrok python-multipart \    rasterio pyproj shapely scipy pillow reportlab \    transformers accelerate bitsandbytes qwen-vl-utilsprint("installed")

In [ ]:
# 2 · Drive + HF cache. Saves 10+ minutes on every reconnect, because the#     7B weights survive a runtime restart instead of downloading again.from google.colab import drivedrive.mount('/content/drive')import osos.environ['HF_HOME'] = '/content/drive/MyDrive/satquery/hf_cache'os.makedirs(os.environ['HF_HOME'], exist_ok=True)# Scenes and previews live here so a restart does not lose uploads.os.environ['SATQUERY_DATA'] = '/content/drive/MyDrive/satquery/data'print('cache:', os.environ['HF_HOME'])

In [ ]:
# 3 · clone (re-run this cell to pull the latest)!rm -rf /content/satquery && git clone https://github.com/<org>/satquery.git /content/satquery%cd /content/satquery/backend

In [ ]:
# 4 · warm the model. Do this BEFORE the demo, not during it.## On OOM, change one string — this is the fallback ladder from the build spec:#   Qwen/Qwen2.5-VL-7B-Instruct  ->  Qwen/Qwen2-VL-7B-Instruct  ->  Qwen/Qwen2-VL-2B-Instructimport osos.environ['SATQUERY_MODEL'] = 'Qwen/Qwen2.5-VL-7B-Instruct'os.environ['SATQUERY_4BIT'] = '1'os.environ['SATQUERY_BACKEND'] = 'auto'   # auto | qwen | stubimport sys; sys.path.insert(0, '/content/satquery/backend')from app.models import qwenqwen.get_model()print(qwen.status())

In [ ]:
# 5 · launchimport nest_asyncio, uvicorn, threadingfrom pyngrok import ngroknest_asyncio.apply()ngrok.kill()url = ngrok.connect(8000)print('PUBLIC URL:', url)print('paste into frontend/.env as VITE_API_URL, and set VITE_MOCK=0')threading.Thread(    target=lambda: uvicorn.run('app.main:app', host='0.0.0.0', port=8000, log_level='info'),    daemon=True,).start()

In [ ]:
# 6 · smoke testimport time; time.sleep(4)!curl -s localhost:8000/health | head -c 600

---## Verify the two mandatory clauses before you rehearseThis runs cross-modal fusion and bi-temporal change against the committed demoscenes, with no model and no frontend. If both print a confidence, the clausesthat decide pass/fail are working.

In [ ]:
# 7 · mandatory-clause checkimport glob, json, requestsS = '/content/satquery/demo/scenes'def run(paths, pair, question):    files = [('files', (p.split('/')[-1], open(p,'rb'), 'image/tiff')) for p in paths]    up = requests.post('http://localhost:8000/upload', files=files,                       data={'pair_type': pair}).json()    print(pair, '·', up['validation']['notes'])    r = requests.post('http://localhost:8000/query',                      json={'scene_ids': up['scene_ids'], 'text': question}, stream=True)    for line in r.iter_lines():        if line and line.startswith(b'data:'):            ev = json.loads(line[5:])            if ev['event'] == 'trace_step':                d = ev['data']                print(f"  {d['step']:02d} {d['label']:<20} {d['confidence']:.2f} {d['confidence_basis']}")            if ev['event'] == 'final':                print('  ->', ev['data']['text'][:220], '\n')run([f'{S}/koyna_s2_2024-07-02_cloud.tif', f'{S}/koyna_s1_2024-07-03_vvvh.tif'],    'cross_modal', 'Is there water under these clouds?')run([f'{S}/koyna_s2_2022-03-09.tif', f'{S}/koyna_s2_2024-03-14.tif'],    'bitemporal', 'What changed between these two dates?')